Import pyspark and Dataset in collab

In [ ]:
!pip install pyspark openpyxl # imort pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Week_5_Task").getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive') # connect google drive to fetch the dataset directly from the drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd # impiort pandas and read datset

pdf = pd.read_excel("/content/drive/MyDrive/Noicy dataset for week 5.xlsx")

In [ ]:
df = spark.createDataFrame(pdf) # store datset in form of tables

In [ ]:
df.show(10) #view records

+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|       city| age|subscription|   status|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   2309|      2025-02-27|  West|           Books|     1382.4|    Houston|32.0|        Free|Completed|2025-02-27 21:47:57|user2309@example.com|user_2309| 100.19|    S092|
|   2330|      2025-12-26| South|        Clothing|    2251.55|    Phoenix|15.0|     Premium|      NaN|2025-12-26 10:17:09|user2330@example.com|user_2330| 311.12|    S006|
|   2494|      2025-08-24|  West|       Groceries|     403.21|    Phoenix|61.0|        Free|  Pending|2025-08-24 22:04:02|user2494@example.com|us

In [ ]:
df.printSchema() # view table schema

root
 |-- user_id: long (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: double (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)



In [ ]:
df.count() # count number of records

10000

PERFORM REQUIRED OPERATIONS ON DATA

In [ ]:
df = df.dropDuplicates(["user_id", "transaction_date"])# Remove duplicate records based on user_id and transaction_date
df.show(7)
print("Total Records After Removing Duplicates:", df.count()) # records after removing duplicates

+-------+----------------+------+----------------+-----------+-------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   city| age|subscription|   status|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45| Boston|66.0|       Basic|Completed|2025-01-21 14:35:00|user1000@example.com|user_1000|1221.09|    S009|
|   1000|      2025-02-28| North|        Clothing|    4487.98| Denver|23.0|       Basic|  Pending|2025-02-28 08:52:47|user1000@example.com|user_1000| 127.58|    S064|
|   1000|      2025-04-06|  East|       Furniture|     768.42|Seattle|18.0|     Premium|  Pending|2025-04-06 04:50:42|user1000@example.com|user_1000|2689.51|    S051

In [ ]:
from pyspark.sql.functions import avg # import Average function
df.filter(df.region == "West")
df.groupBy("product_category")
df.agg(avg("sale_amount").alias("average_sale_amount"))  # calculate average sale amount
df.show(7)

+-------+----------------+------+----------------+-----------+-------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   city| age|subscription|   status|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45| Boston|66.0|       Basic|Completed|2025-01-21 14:35:00|user1000@example.com|user_1000|1221.09|    S009|
|   1000|      2025-02-28| North|        Clothing|    4487.98| Denver|23.0|       Basic|  Pending|2025-02-28 08:52:47|user1000@example.com|user_1000| 127.58|    S064|
|   1000|      2025-04-06|  East|       Furniture|     768.42|Seattle|18.0|     Premium|  Pending|2025-04-06 04:50:42|user1000@example.com|user_1000|2689.51|    S051

In [ ]:
from pyspark.sql.functions import when, col  # the coulumn contains NaN so use this method to convert status from null to unkown
df = df.withColumn("status",when(col("status").isNull() | (col("status") == "NaN"), "Unknown").otherwise(col("status")))
df.show(20)

+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|       city| age|subscription|   status|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45|     Boston|66.0|       Basic|Completed|2025-01-21 14:35:00|user1000@example.com|user_1000|1221.09|    S009|
|   1000|      2025-02-28| North|        Clothing|    4487.98|     Denver|23.0|       Basic|  Pending|2025-02-28 08:52:47|user1000@example.com|user_1000| 127.58|    S064|
|   1000|      2025-04-06|  East|       Furniture|     768.42|    Seattle|18.0|     Premium|  Pending|2025-04-06 04:50:42|user1000@example.com|us

In [ ]:
from pyspark.sql.functions import count  # import count function

city_count = df.groupBy("city") \
               .agg(count("city").alias("city_count")) \
               .filter("city_count > 100")   # Display cities whose count is above 100

city_count.show(7)

+-----------+----------+
|       city|city_count|
+-----------+----------+
|    Phoenix|      1040|
|     Dallas|       970|
|Los Angeles|      1010|
|    Chicago|       963|
|    Seattle|       987|
|    Houston|       961|
|      Miami|      1023|
+-----------+----------+
only showing top 7 rows


In [ ]:
clean_df = df.drop("status") #Drop the status column
clean_df.show(7)

+-------+----------------+------+----------------+-----------+-------+----+------------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   city| age|subscription|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-------+----+------------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45| Boston|66.0|       Basic|2025-01-21 14:35:00|user1000@example.com|user_1000|1221.09|    S009|
|   1000|      2025-02-28| North|        Clothing|    4487.98| Denver|23.0|       Basic|2025-02-28 08:52:47|user1000@example.com|user_1000| 127.58|    S064|
|   1000|      2025-04-06|  East|       Furniture|     768.42|Seattle|18.0|     Premium|2025-04-06 04:50:42|user1000@example.com|user_1000|2689.51|    S051|
|   1000|      2025-04-15|  East|          Sports|     230

In [ ]:
clean_df = df.withColumnRenamed("raw_timestamp", "event_time") # rename the column
clean_df.show(7)

+-------+----------------+------+----------------+-----------+-------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   city| age|subscription|   status|         event_time|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45| Boston|66.0|       Basic|Completed|2025-01-21 14:35:00|user1000@example.com|user_1000|1221.09|    S009|
|   1000|      2025-02-28| North|        Clothing|    4487.98| Denver|23.0|       Basic|  Pending|2025-02-28 08:52:47|user1000@example.com|user_1000| 127.58|    S064|
|   1000|      2025-04-06|  East|       Furniture|     768.42|Seattle|18.0|     Premium|  Pending|2025-04-06 04:50:42|user1000@example.com|user_1000|2689.51|    S051

In [ ]:
users = df.filter((df.age >= 18) &(df.age <= 30) &(df.subscription == "Premium")) # filter records where age is between 18 to 30 and subscription is premium
users.show(7)

+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|       city| age|subscription|   status|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-04-06|  East|       Furniture|     768.42|    Seattle|18.0|     Premium|  Pending|2025-04-06 04:50:42|user1000@example.com|user_1000|2689.51|    S051|
|   1002|      2025-10-03| South|       Groceries|    2385.77|Los Angeles|21.0|     Premium|  Pending|2025-10-03 05:10:46|user1002@example.com|user_1002| 864.38|    S060|
|   1003|      2025-06-28| North|       Furniture|     401.47|     Denver|19.0|     Premium|Cancelled|2025-06-28 20:18:42|user1003@example.com|us

In [ ]:
df = df.na.fill({"price": 0}) # convert prize to 0 if it has null
df.show(20)

+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|       city| age|subscription|   status|      raw_timestamp|               email| username|  price|store_id|
+-------+----------------+------+----------------+-----------+-----------+----+------------+---------+-------------------+--------------------+---------+-------+--------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45|     Boston|66.0|       Basic|Completed|2025-01-21 14:35:00|user1000@example.com|user_1000|1221.09|    S009|
|   1000|      2025-02-28| North|        Clothing|    4487.98|     Denver|23.0|       Basic|  Pending|2025-02-28 08:52:47|user1000@example.com|user_1000| 127.58|    S064|
|   1000|      2025-04-06|  East|       Furniture|     768.42|    Seattle|18.0|     Premium|  Pending|2025-04-06 04:50:42|user1000@example.com|us

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType
df = df.withColumn("event_time",col("raw_timestamp").cast(TimestampType())).drop("raw_timestamp")
# create a new column and convert string to timestanp format then drop the old column and display records
df.show(7)

+-------+----------------+------+----------------+-----------+-------+----+------------+---------+--------------------+---------+-------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city| age|subscription|   status|               email| username|  price|store_id|         event_time|
+-------+----------------+------+----------------+-----------+-------+----+------------+---------+--------------------+---------+-------+--------+-------------------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45| Boston|66.0|       Basic|Completed|user1000@example.com|user_1000|1221.09|    S009|2025-01-21 14:35:00|
|   1000|      2025-02-28| North|        Clothing|    4487.98| Denver|23.0|       Basic|  Pending|user1000@example.com|user_1000| 127.58|    S064|2025-02-28 08:52:47|
|   1000|      2025-04-06|  East|       Furniture|     768.42|Seattle|18.0|     Premium|  Pending|user1000@example.com|user_1000|2689.51|    S051|2025-04-06 04:50:42

In [ ]:
# shuffle operation
df.groupBy("city").count().show()

+-----------+-----+
|       city|count|
+-----------+-----+
|    Phoenix| 1040|
|     Dallas|  970|
|Los Angeles| 1010|
|    Chicago|  963|
|    Seattle|  987|
|    Houston|  961|
|      Miami| 1023|
|   New York|  986|
|     Denver|  988|
|     Boston|  995|
+-----------+-----+



In [ ]:
from pyspark.sql.functions import col
clean_df = df.filter(col("email").isNotNull() &(col("username") != "")) # show columns where email and username is not null
clean_df.show(7)

+-------+----------------+------+----------------+-----------+-------+----+------------+---------+--------------------+---------+-------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   city| age|subscription|   status|               email| username|  price|store_id|         event_time|
+-------+----------------+------+----------------+-----------+-------+----+------------+---------+--------------------+---------+-------+--------+-------------------+
|   1000|      2025-01-21|  West|        Clothing|    4480.45| Boston|66.0|       Basic|Completed|user1000@example.com|user_1000|1221.09|    S009|2025-01-21 14:35:00|
|   1000|      2025-02-28| North|        Clothing|    4487.98| Denver|23.0|       Basic|  Pending|user1000@example.com|user_1000| 127.58|    S064|2025-02-28 08:52:47|
|   1000|      2025-04-06|  East|       Furniture|     768.42|Seattle|18.0|     Premium|  Pending|user1000@example.com|user_1000|2689.51|    S051|2025-04-06 04:50:42

In [ ]:
from pyspark.sql.functions import min, max, mean
df.agg(min("price").alias("Minimum_Price"),max("price").alias("Maximum_Price"),mean("price").alias("Average_Price")).show()
# show minimun, maximum and average price

+-------------+-------------+-----------------+
|Minimum_Price|Maximum_Price|    Average_Price|
+-------------+-------------+-----------------+
|          0.0|      2999.77|1414.862150559308|
+-------------+-------------+-----------------+



In [ ]:
from pyspark.sql.functions import sum
# remove duplicates and fill null prices to 0 also calculate total revenue
final_df = df.dropDuplicates() \
             .na.fill({"price": 0}) \
             .groupBy("store_id") \
             .agg(sum("price").alias("total_revenue"))

final_df.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S066|         140614.16|
|    S032|168864.70999999996|
|    S076|         151563.08|
|    S089|185051.71000000005|
|    S084|142381.61999999994|
|    S023|126737.81999999999|
|    S025|127754.54999999997|
|    S067| 169688.3500000001|
|    S028|137837.78999999998|
|    S035|134091.48999999996|
|    S004|140960.93000000002|
|    S033|         121352.34|
|    S078|         159848.26|
|    S001|         140317.22|
|    S083|162652.59000000005|
|    S027|123728.19000000002|
|    S016|137834.10999999996|
|    S024|146367.62999999998|
|    S008|130467.44000000003|
|    S072|164140.44999999998|
+--------+------------------+
only showing top 20 rows
